# Pandas — Phase 5: Aggregation & Grouping
### Credit Card Risk Analysis Track — Reporting Insights

**Topics in this phase:**
21. Basic Aggregations — `.mean()`, `.median()`, `.sum()`, `.value_counts()`
22. GroupBy Summaries — risk metrics by category
23. Multi-Metric Aggregations — `.groupby().agg()`
24. Pivot Tables — spreadsheet-style cross-referencing

**Dataset:** the clean `loan_applications.csv` from Phase 1, same as Phase 4. The setup cell below also adds two synthetic columns the original export didn't have — `loan_purpose` (needed for the two-way pivot tables in Topic 24) and `risk_tier` (reused from the Phase 4-style credit tiering, needed for Topic 23) — plus `was_approved`, a binary version of `loan_status`. Keep `loan_applications.csv` in the same folder as this notebook.

**How to use this notebook:**
- Each question has a `YOUR CODE HERE` cell — attempt it first.
- The `Solution` cell right after shows one correct approach — compare, don't just copy.
- All solutions were run against the actual dataset before this notebook was assembled.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("loan_applications.csv", parse_dates=["application_date"])
df["annual_income"] = df["annual_income"].fillna(df["annual_income"].median())
df["credit_score"] = df["credit_score"].fillna(df["credit_score"].median())

rng = np.random.default_rng(55)
df["loan_purpose"] = rng.choice(
    ["Debt Consolidation", "Home Improvement", "Medical", "Auto", "Education", "Business"],
    size=len(df),
)

df["risk_tier"] = np.select(
    [df["credit_score"] >= 700, df["credit_score"] >= 620],
    ["Prime", "Near-Prime"],
    default="Subprime",
)
df["was_approved"] = np.where(df["loan_status"] == "Approved", 1, 0)

print(df.shape)
df.head()

## Topic 21: Basic Aggregations

A portfolio-level report usually starts with a handful of single numbers — the "how are we doing overall" view.

**Q1.** Compute `avg_loan_amount`, the mean of `loan_amount` across the whole portfolio.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
avg_loan_amount = df["loan_amount"].mean()
print(avg_loan_amount)

**Q2.** Compute `median_income`, the median of `annual_income`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
median_income = df["annual_income"].median()
print(median_income)

**Q3.** Compute `total_existing_debt`, the sum of `existing_debt` across every borrower — the total debt the portfolio's applicants are already carrying, outside of these loan applications.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
total_existing_debt = df["existing_debt"].sum()
print(total_existing_debt)

**Q4.** Get `loan_status_counts`, how many applications fall into each status, using `.value_counts()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
loan_status_counts = df["loan_status"].value_counts()
print(loan_status_counts)

**Q5.** Get `home_ownership_pct`: the **percentage** breakdown of `home_ownership` (not raw counts), using `.value_counts(normalize=True)`, then multiplying by 100 and rounding to 4 decimals.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
home_ownership_pct = df["home_ownership"].value_counts(normalize=True).round(4) * 100
print(home_ownership_pct)

**Q6.** Compute `credit_score_std`, the standard deviation of `credit_score` — how spread out the portfolio's credit quality is.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
credit_score_std = df["credit_score"].std()
print(credit_score_std)

## Topic 22: GroupBy Summaries

Portfolio-wide numbers hide differences between segments. GroupBy is how you break a metric apart by category.

**Q7.** Compute `avg_score_by_employment`: the mean `credit_score` for each `employment_status` group, using `.groupby("employment_status")["credit_score"].mean()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
avg_score_by_employment = df.groupby("employment_status")["credit_score"].mean()
print(avg_score_by_employment)

**Q8.** Compute `avg_loan_by_status`: the mean `loan_amount` for each `loan_status`. Does the average loan size for `"Denied"` applications look different from `"Approved"`?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
avg_loan_by_status = df.groupby("loan_status")["loan_amount"].mean()
print(avg_loan_by_status)

**Q9.** Compute `count_by_tier`: how many applications fall into each `risk_tier`, using `.groupby("risk_tier").size()` (note: `.size()` counts rows including any nulls in other columns; `.count()` on a specific column would exclude nulls in *that* column — usually `.size()` is what you want for "how many rows in this group").

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
count_by_tier = df.groupby("risk_tier").size()
print(count_by_tier)

**Q10.** Compute `avg_income_by_ownership`: mean `annual_income` per `home_ownership` group, sorted **highest to lowest** (chain `.sort_values(ascending=False)` onto the groupby result).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
avg_income_by_ownership = df.groupby("home_ownership")["annual_income"].mean().sort_values(ascending=False)
print(avg_income_by_ownership)

**Q11.** Compute `approval_rate_by_employment`: the mean of `was_approved` per `employment_status`. (This is a common trick — the mean of a 0/1 flag *is* the approval rate, since it's just "fraction of 1s.")

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
approval_rate_by_employment = df.groupby("employment_status")["was_approved"].mean()
print(approval_rate_by_employment)

**Q12.** Group by **two** columns at once — `employment_status` and `home_ownership` together — and compute the mean `loan_amount` for each combination, into `avg_loan_by_emp_home` (pass a list to `.groupby()`).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
avg_loan_by_emp_home = df.groupby(["employment_status", "home_ownership"])["loan_amount"].mean()
print(avg_loan_by_emp_home)

## Topic 23: Multi-Metric Aggregations

Often you want several statistics per group at once, not just one — `.agg()` is how.

**Q13.** For each `risk_tier`, compute the min, max, and mean of `loan_amount` **in one call**, into `loan_stats_by_tier`, using `.groupby("risk_tier")["loan_amount"].agg(["min", "max", "mean"])`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
loan_stats_by_tier = df.groupby("risk_tier")["loan_amount"].agg(["min", "max", "mean"])
print(loan_stats_by_tier)

**Q14.** Go further: for each `risk_tier`, get min/max/mean of `loan_amount` **and** the mean of `credit_score`, all in one `.agg()` call, into `multi_col_agg`, by passing a dict `{"loan_amount": [...], "credit_score": "mean"}`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
multi_col_agg = df.groupby("risk_tier").agg({
    "loan_amount": ["min", "max", "mean"],
    "credit_score": "mean",
})
print(multi_col_agg)

**Q15.** The dict-of-lists approach from Q14 produces awkward multi-level column names. Redo it with **named aggregation** instead: `df.groupby("risk_tier").agg(min_loan=("loan_amount", "min"), max_loan=("loan_amount", "max"), avg_loan=("loan_amount", "mean"), avg_score=("credit_score", "mean"))`, into `named_agg`. Compare its column structure to `multi_col_agg` — this is the cleaner, more readable style for reports.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
named_agg = df.groupby("risk_tier").agg(
    min_loan=("loan_amount", "min"),
    max_loan=("loan_amount", "max"),
    avg_loan=("loan_amount", "mean"),
    avg_score=("credit_score", "mean"),
)
print(named_agg)

**Q16.** Using named aggregation, build `employment_summary`: for each `employment_status`, get `application_count` (count of `application_id`) and `total_loan_amount` (sum of `loan_amount`).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
employment_summary = df.groupby("employment_status").agg(
    application_count=("application_id", "count"),
    total_loan_amount=("loan_amount", "sum"),
)
print(employment_summary)

**Q17.** A `.groupby()` result uses the grouped column as its **index**, which is awkward if you want to treat it like a normal table (e.g. to sort, filter, or export). Take `loan_stats_by_tier` from Q13 and chain `.reset_index()` onto it, into `tier_summary_flat`. Print the result and confirm it's a regular DataFrame with `risk_tier` as a normal column again.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
tier_summary_flat = df.groupby("risk_tier")["loan_amount"].agg(["min", "max", "mean"]).reset_index()
print(tier_summary_flat)
print(type(tier_summary_flat))

**Q18.** Now that `tier_summary_flat` is a normal DataFrame, sort it by the `mean` column descending, into `ranked_tiers` — which risk tier actually has the largest average loan size? (You may be surprised it isn't a strict Prime→Subprime ordering.)

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
ranked_tiers = tier_summary_flat.sort_values("mean", ascending=False)
print(ranked_tiers)

## Topic 24: Pivot Tables

A pivot table is a groupby on **two** dimensions at once, laid out like a spreadsheet: one category down the rows, another across the columns.

**Q19.** Build `pivot_avg_loan`: a pivot table with `home_ownership` as the row index, `loan_purpose` as the columns, and the **mean** `loan_amount` as the values, using `df.pivot_table(index=..., columns=..., values=..., aggfunc="mean")`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
pivot_avg_loan = df.pivot_table(
    index="home_ownership", columns="loan_purpose", values="loan_amount", aggfunc="mean"
)
print(pivot_avg_loan.round(0))

**Q20.** Build `pivot_counts`: the same layout, but counting applications instead of averaging — use `values="application_id", aggfunc="count"`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
pivot_counts = df.pivot_table(
    index="home_ownership", columns="loan_purpose", values="application_id", aggfunc="count"
)
print(pivot_counts)

**Q21.** Rebuild the Q19 pivot table but add `margins=True`, into `pivot_with_margins` — this adds an `"All"` row and column showing the overall average for each row/column, plus the grand total.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
pivot_with_margins = df.pivot_table(
    index="home_ownership", columns="loan_purpose", values="loan_amount",
    aggfunc="mean", margins=True
)
print(pivot_with_margins.round(0))

**Q22.** For a **proportion**-based cross-tab instead of a pivot on a numeric column, use `pd.crosstab(df["home_ownership"], df["loan_status"], normalize="index")` into `status_crosstab` — each row now shows what fraction of that ownership group falls into each loan status (rows sum to 1).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
status_crosstab = pd.crosstab(df["home_ownership"], df["loan_status"], normalize="index").round(3)
print(status_crosstab)

**Q23.** Build `pivot_multi_values`: `risk_tier` as the index, `loan_purpose` as the columns, but this time aggregate **two** value columns at once — pass `values=["loan_amount", "credit_score"]` — both averaged.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
pivot_multi_values = df.pivot_table(
    index="risk_tier", columns="loan_purpose", values=["loan_amount", "credit_score"], aggfunc="mean"
)
print(pivot_multi_values.round(0))

**Q24.** Build `pivot_filled`: `employment_status` as index, `loan_purpose` as columns, mean `loan_amount` as values — but this time pass `fill_value=0`. Some employment/purpose combinations may have very few or zero applications; without `fill_value`, those cells would show `NaN`, which can be misleading in a report meant to be read at a glance.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
pivot_filled = df.pivot_table(
    index="employment_status", columns="loan_purpose", values="loan_amount",
    aggfunc="mean", fill_value=0
)
print(pivot_filled.round(0))

## ✅ Checkpoint

**What you covered:**
- Basic aggregations: `.mean()`, `.median()`, `.sum()`, `.std()`, `.value_counts()` (raw and `normalize=True`)
- GroupBy summaries: single- and multi-column `.groupby()`, `.size()` vs `.count()`, sorting a groupby result, and the "mean of a 0/1 flag = rate" trick
- Multi-metric aggregation: `.agg([...])` with a list, `.agg({...})` with a dict, cleaner **named aggregation**, and `.reset_index()` to flatten a groupby result back into a normal DataFrame
- Pivot tables: `.pivot_table()` with `aggfunc="mean"`/`"count"`, `margins=True` for row/column totals, multiple `values=` columns at once, `fill_value=` for missing combinations, and `pd.crosstab(..., normalize="index")` for proportion-based cross-tabs

**Why it matters for the project:** this is how raw row-level data becomes an actual portfolio report — "what's our approval rate by employment type," "which risk tier carries the largest loans," "how does loan purpose vary by home ownership" are exactly the questions a credit risk team asks, and exactly what this phase answers.

**What's next:** Phase 6, whenever you're ready — let me know the topics you want covered.